In [ ]:

from config.config import *
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType

dim_country = spark.read.format("delta").load(f"{gold_path}/dim_country")
dim_date = spark.read.format("delta").load(f"{gold_path}/dim_date")

def join_country(df, colname="country_key"):
    return df.join(dim_country, on=colname, how="left")

# 1) Yearly arrivals by country
fa = spark.read.format("delta").load(f"{gold_path}/fact_tourism_arrivals")
gold_yearly_arrivals_by_country = (
    fa.filter(F.col("frequency") == "A")
      .groupBy("country_key", "geo_code", "country_name", "year", "source_system", "indicator_code", "visitor_category")
      .agg(F.sum("standardized_value").alias("arrivals"), F.first("standardized_unit").alias("unit"))
      .withColumn("date_key", F.col("year"))
      .join(dim_date, on="date_key", how="left")
      .select("country_key", "geo_code", "country_name", "year", "source_system", "indicator_code",
              "visitor_category", "arrivals", "unit", "is_primary_period")
)
gold_yearly_arrivals_by_country.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/gold_yearly_arrivals_by_country")

# 2) Monthly arrivals by country
gold_monthly_arrivals_by_country = (
    fa.filter(F.col("frequency") == "M")
      .groupBy("country_key", "geo_code", "country_name", "year", "month", "source_system", "indicator_code", "visitor_category")
      .agg(F.sum("standardized_value").alias("arrivals"), F.first("standardized_unit").alias("unit"))
      .join(dim_date, on=[dim_date["year"] == F.col("year"), dim_date["month"] == F.col("month")], how="left")
      .select("country_key", "geo_code", "country_name", "year", "month", "source_system", "indicator_code",
              "visitor_category", "arrivals", "unit", "is_primary_period")
)
gold_monthly_arrivals_by_country.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/gold_monthly_arrivals_by_country")
print("yearly arrivals:", gold_yearly_arrivals_by_country.count(), "monthly arrivals:", gold_monthly_arrivals_by_country.count())


In [ ]:

from config.config import *
from pyspark.sql import functions as F

dim_country = spark.read.format("delta").load(f"{gold_path}/dim_country")
dim_date = spark.read.format("delta").load(f"{gold_path}/dim_date")

# Year-over-year growth (yearly)
y = spark.read.format("delta").load(f"{gold_path}/gold_yearly_arrivals_by_country")
y_cur = y.select("country_key", "year", "source_system", "indicator_code", "visitor_category", "arrivals")
y_prev = y_cur.withColumnRenamed("year", "prev_year").withColumnRenamed("arrivals", "prev_arrivals")
gold_tourism_growth_yearly = (
    y_cur.join(y_prev, on=["country_key", "source_system", "indicator_code", "visitor_category"], how="inner")
         .filter(F.col("year") == F.col("prev_year") + 1)
         .withColumn("growth_pct", (F.col("arrivals") - F.col("prev_arrivals")) / F.when(F.col("prev_arrivals") != 0, F.col("prev_arrivals")))
         .select("country_key", "year", "source_system", "indicator_code", "visitor_category", "arrivals", "prev_arrivals", "growth_pct")
         .join(dim_country, on="country_key", how="left")
)
gold_tourism_growth_yearly.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/gold_tourism_growth_yearly")

# Year-over-year monthly growth (same month previous year)
m = spark.read.format("delta").load(f"{gold_path}/gold_monthly_arrivals_by_country")
m_cur = m.select("country_key", "year", "month", "source_system", "indicator_code", "visitor_category", "arrivals")
m_prev = m_cur.withColumnRenamed("year", "prev_year").withColumnRenamed("arrivals", "prev_arrivals")
gold_tourism_growth_monthly = (
    m_cur.join(m_prev, on=["country_key", "month", "source_system", "indicator_code", "visitor_category"], how="inner")
         .filter(F.col("year") == F.col("prev_year") + 1)
         .withColumn("growth_pct", (F.col("arrivals") - F.col("prev_arrivals")) / F.when(F.col("prev_arrivals") != 0, F.col("prev_arrivals")))
         .select("country_key", "year", "month", "source_system", "indicator_code", "visitor_category", "arrivals", "prev_arrivals", "growth_pct")
         .join(dim_country, on="country_key", how="left")
)
gold_tourism_growth_monthly.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/gold_tourism_growth_monthly")
print("yearly growth:", gold_tourism_growth_yearly.count(), "monthly growth:", gold_tourism_growth_monthly.count())


In [ ]:

from config.config import *
from pyspark.sql import functions as F

dim_country = spark.read.format("delta").load(f"{gold_path}/dim_country")
y = spark.read.format("delta").load(f"{gold_path}/gold_yearly_arrivals_by_country").filter(F.col("is_primary_period") == True)
# Top destinations by absolute arrivals (yearly)
gold_destination_rankings = (
    y.groupBy("country_key", "geo_code", "country_name")
     .agg(F.sum("arrivals").alias("total_arrivals"))
     .orderBy(F.desc("total_arrivals"))
     .limit(50)
     .join(dim_country, on="country_key", how="left")
)
gold_destination_rankings.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/gold_destination_rankings")

# Seasonality from monthly arrivals (average/median by month across all years)
m = spark.read.format("delta").load(f"{gold_path}/gold_monthly_arrivals_by_country")
gold_tourism_seasonality = (
    m.groupBy("month").agg(
        F.avg("arrivals").alias("avg_arrivals"),
        F.median("arrivals").alias("median_arrivals"),
        F.max("arrivals").alias("peak_arrivals"),
        F.min("arrivals").alias("lowest_arrivals"),
        F.stddev_pop("arrivals").alias("stddev"),
        F.max("arrivals") / F.when(F.avg("arrivals") != 0, F.avg("arrivals")).alias("peak_index"),
    )
    .orderBy("month")
)
gold_tourism_seasonality.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/gold_tourism_seasonality")
print("rankings:", gold_destination_rankings.count(), "seasonality rows:", gold_tourism_seasonality.count())


In [ ]:

from config.config import *
from pyspark.sql import functions as F

fn = spark.read.format("delta").load(f"{gold_path}/fact_nights_spent")
gold_nights_spent_trends = (
    fn.groupBy("country_key", "year", "is_primary_period")
      .agg(F.sum("standardized_value").alias("total_nights"))
      .orderBy("country_key", "year")
      .withColumn("nights_per_month", F.col("total_nights") / F.lit(12))
)
gold_nights_spent_trends.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/gold_nights_spent_trends")

hp = spark.read.format("delta").load(f"{gold_path}/fact_hotel_performance")
gold_hotel_capacity_trends = hp.groupBy("country_key", "year", "accommodation_metric").agg(
    F.sum("standardized_value").alias("total_units"), F.first("original_unit").alias("unit")
).orderBy("country_key", "year")
gold_hotel_capacity_trends.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/gold_hotel_capacity_trends")

occ = spark.read.format("delta").load(f"{gold_path}/fact_hotel_occupancy")
gold_hotel_occupancy_trends = occ.groupBy("country_key", "year", "month").agg(
    F.avg("occupancy_rate").alias("avg_occupancy_rate")
).orderBy("country_key", "year", "month")
gold_hotel_occupancy_trends.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/gold_hotel_occupancy_trends")
print("nights:", gold_nights_spent_trends.count(), "hotel_capacity:", gold_hotel_capacity_trends.count(), "occupancy:", gold_hotel_occupancy_trends.count())


In [ ]:

from config.config import *
from pyspark.sql import functions as F

# Visitor category analysis: nightly vs excursionist composition by country and year
fa = spark.read.format("delta").load(f"{gold_path}/fact_tourism_arrivals")
gold_visitor_category_analysis = (
    fa.filter(F.col("frequency") == "A")
      .groupBy("country_key", "country_name", "year", "visitor_category")
      .agg(F.sum("standardized_value").alias("arrivals"))
      .orderBy("country_key", "year", "visitor_category")
)
gold_visitor_category_analysis.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/gold_visitor_category_analysis")

# Country comparison: yearly arrivals by source
y = spark.read.format("delta").load(f"{gold_path}/gold_yearly_arrivals_by_country")
gold_country_comparison = y.groupBy("country_key", "country_name", "source_system").agg(F.sum("arrivals").alias("total_arrivals"))
gold_country_comparison.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/gold_country_comparison")
print("visitor:", gold_visitor_category_analysis.count(), "country:", gold_country_comparison.count())


In [ ]:

from config.config import *
from pyspark.sql import functions as F

# Source comparison: World Bank vs UN vs Eurostat arrivals. Uses normalized units and excludes aggregates.
wb = spark.read.format("delta").load(f"{silver_path}/world_bank_international_arrivals").filter(F.col("country_iso3").isNotNull())
un = spark.read.format("delta").load(f"{silver_path}/un_tourism_inbound_arrivals").filter(F.col("reporter_country_iso3").isNotNull())
estat_arm = spark.read.format("delta").load(f"{silver_path}/estat_tour_occ_arm").filter(F.col("unit_code") == "NR" and F.col("country_iso3").isNotNull())

def summary(df, col, source):
    return df.groupBy(col).agg(
        F.sum(df[col].alias("country")).alias("country"),
        F.count("*").alias("rows"),
        F.avg("value").alias("avg_value"),
        F.stddev_pop("value").alias("stddev")
    )

wb_s = wb.groupBy("country_code").agg(F.sum(F.col("arrivals") / 1000.0).alias("world_bank_arrivals"), F.count("*").alias("world_bank_rows"))
un_s = un.groupBy("reporter_country_iso3").agg(F.sum(F.col("arrivals") / 1000.0).alias("un_arrivals"), F.count("*").alias("un_rows"))
estat_s = estat_arm.groupBy("country_iso3").agg(F.sum(F.col("tourism_value")).alias("estat_arrivals"), F.count("*").alias("estat_rows"))

gold_source_comparison = (
    wb_s.join(un_s, wb_s["country_code"] == un_s["reporter_country_iso3"], how="full_outer")
        .join(estat_s, F.coalesce(wb_s["country_code"], un_s["reporter_country_iso3"]) == estat_s["country_iso3"], how="full_outer")
        .select(
            F.coalesce(wb_s["country_code"], un_s["reporter_country_iso3"], estat_s["country_iso3"]).alias("country_code"),
            F.coalesce(wb_s["world_bank_arrivals"], F.lit(None).cast("double")).alias("world_bank_arrivals"),
            F.coalesce(wb_s["world_bank_rows"], F.lit(0)).alias("world_bank_rows"),
            F.coalesce(un_s["un_arrivals"], F.lit(None).cast("double")).alias("un_arrivals"),
            F.coalesce(un_s["un_rows"], F.lit(0)).alias("un_rows"),
            F.coalesce(estat_s["estat_arrivals"], F.lit(None).cast("double")).alias("estat_arrivals"),
            F.coalesce(estat_s["estat_rows"], F.lit(0)).alias("estat_rows"),
        )
        .withColumn("discrepancy", F.greatest(F.col("world_bank_arrivals"), F.col("un_arrivals"), F.col("estat_arrivals")))
        .orderBy(F.desc("discrepancy"), "country_code")
        .limit(100)
)
gold_source_comparison.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/gold_source_comparison")
print("source comparison:", gold_source_comparison.count())
